In [1]:
import os, glob
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

In [2]:
pth = os.path.join('..', '..')
pth_data = os.path.join(pth, 'data')
pth_in = os.path.join(pth_data, 'development', 'modeling')
pth_feat = os.path.join(pth_data, 'features', 'biomart')
# pth_fig = os.path.join(pth, 'figures', 'development', 'signatures')
# pth_out = os.path.join(pth_feat, 'modules')

In [4]:
adata_bif = sc.read_h5ad(os.path.join(pth_in, 'landscape.h5ad'))
adata = adata_bif[adata_bif.obs.source == 'GSE162534'].copy()
adata

AnnData object with n_obs × n_vars = 2811 × 510
    obs: 'sample', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden', 'celltype', 'total_counts_ribo', 'pct_counts_ribo', 'T+ Sox2+', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.1', 'seurat_clusters', 'age', 'RNA_snn_res.0.5', 'ident', 'velocity_self_transition', 'celltype_original', 'source', 't_orig', 'early', 'late', 'spectrum', 't', 'seg', 'edge', 'milestones'
    uns: 'celltype_colors', 'graph', 'milestones_colors', 'pseudotime_list', 'seg_colors'
    obsm: 'X_R', 'X_mesencoder', 'X_pca', 'X_pca_harmony', 'X_spectrum', 'X_umap', 'X_umap_traj', 'velocity_umap'
    layers: 'logcounts', 'scaled'

In [ ]:
feat_fn = sorted(glob.glob(os.path.join(pth_feat, '*.csv')))
get_src = lambda fn: os.path.split(fn)[1].replace('.csv', '')
feat_dict = {get_src(fn) : pd.read_csv(fn) for fn in feat_fn}

# module signatures
for src, df_src in feat_dict.items():
    if src != 'union':
        name = names_dict[src]
        print(name)
        g = df_src.mmusculus.copy()
        sc.tl.score_genes(adata1,
                          gene_list = g,
                          score_name = name,
                          ctrl_as_ref = False,
                          random_state = 1234)

In [ ]:
adata1.obs['t_seg'] = pd.cut(adata1.obsm['X_mesencoder'][:, 1], 8, labels = False)
adata1.obs.t_seg

In [ ]:
g_early = pd.merge(left = feat_dict['GSE162534'],
                   right = feat_dict['GSE229103'],
                   how = 'outer',
                   indicator = 'module')
g_early['module'] = g_early.module.map({
    'left_only'  : 'Mes1',
    'right_only' : 'Mes2',
    'both'       : 'Mes1/2'})

g_early = g_early.loc[g_early.module != 'Mes1/2'].reset_index()

g_early

In [ ]:
c_dict = {'Mes1' : 'r', 'Mes2' : 'b', 'Mes1/2' : 'purple'}

fig, ax = plt.subplots(1, 8, figsize = (8, 1.5), sharex = True, sharey = True)
for seg, sdf in adata1.obs.groupby('t_seg', observed = True):
    X = adata1[sdf.index, g_early.mmusculus].X.toarray()
    for ix in g_early.index:
        for i in range(1, 3):
            g_early.loc[ix, f'R_Mes{i}_{seg}'] = np.corrcoef(X[:, ix], sdf[f'Mes{i}'])[0, 1]
    
    ax[seg].scatter(*g_early[[f'R_Mes1_{seg}', f'R_Mes2_{seg}']].values.T, c = g_early.module.map(c_dict), s = 1)
    ax[seg].set_title(g_early[[f'R_Mes1_{seg}', f'R_Mes2_{seg}']].corr().iloc[0, 1].round(2))

    fig_sc, ax_sc = plt.subplots(1, 1, figsize = (3.5, 1.5))
    sc.pl.embedding(adata, 'X_mesencoder', dimensions = [1, 0],
                    size = 70, alpha = .5, show = False, ax = ax_sc)
    
    sc.pl.embedding(adata1[sdf.index], 'X_mesencoder', dimensions = [1, 0],
                    na_color = 'darkgrey', size = 70, alpha = .5,
                    ax = ax_sc, show = False)
    
    ax_sc.invert_yaxis()
    fig_sc.show()
    


fig.tight_layout()
fig.show()


# g_early



In [ ]:
sc.pl.embedding(adata, 'X_mesencoder', dimensions = [1, 0],
                color = 'celltype', size = 100, show = False)
plt.gca().invert_yaxis(); plt.show()

In [ ]:
scf.tl.tree(adata, method = 'ppt', Nodes = 100, use_rep = 'X_mesencoder',
            ppt_lambda = 100, ppt_sigma = 0.3, ppt_nsteps = 200, seed = 1,
            plot = True, basis = 'mesencoder')

In [ ]:
scf.tl.root(adata, 84)

In [ ]:
scf.tl.pseudotime(adata)

In [ ]:
sc.pl.embedding(adata, 'X_mesencoder', dimensions = [1, 0],
                color = 't', size = 100, show = False)
plt.gca().invert_yaxis(); plt.show()

In [ ]:
sc.pl.embedding(adata, 'X_mesencoder', dimensions = [1, 0],
                color = 'milestones', size = 100, show = False)
plt.gca().invert_yaxis(); plt.show()

In [ ]:
scf.tl.rename_milestones(adata, new = {'0'  : 'Early2',
                                       '21' : 'NE',
                                       '64' : 'Late2',
                                       '8'  : 'Bif',
                                       '84' : 'Root'})
scf.pl.milestones(adata, basis = 'mesencoder', annotate = True)

In [ ]:
scf.tl.test_association(adata, n_jobs = 8)

In [ ]:
scf.tl.test_association(adata, reapply_filters = True, A_cut = .5)

In [ ]:
scf.tl.fit(adata, n_jobs = 8)

In [ ]:
scf.tl.test_fork(adata, root_milestone = 'Root', milestones = ['NE', 'Late2'], n_jobs = 8, rescale = True)

In [ ]:
scf.tl.branch_specific(adata, root_milestone = 'Root', milestones = ['NE', 'Late2'], effect = .5)

In [ ]:
g1 = scf.pl.trends(adata, root_milestone = 'Root', milestones = ['NE', 'Late2'], branch = 'NE',
                   plot_emb = False, figsize = (6, 5.5), ordering = 'quantile', return_genes = True)

# ax = scf.pl.trends(adata, features = adata.var_names,
#                    ordering = 'quantile', n_features = 25,
#                    offset_names = .1, fontsize = 8,
#                    style = 'italic', plot_emb = False,
#                    figsize = (6, 5.5), show = False)

In [ ]:
g2 = scf.pl.trends(adata, root_milestone = 'Root', milestones = ['NE', 'Late2'], branch = 'Late2',
                   plot_emb = False, figsize = (6, 5.5), ordering = 'quantile', return_genes = True)

In [ ]:
scf.pl.matrix(adata, g1.tolist() + g2.tolist(), norm = 'minmax', cmap = 'RdBu_r', colorbar = False)

In [ ]:
scf.tl.activation_lm(adata, root_milestone = 'Root', milestones = ['NE', 'Late2'], n_jobs = 8)

In [ ]:
scf.tl.slide_cells(adata, root_milestone = 'Root', milestones = ['NE', 'Late2'], win = 200)

In [ ]:
scf.tl.slide_cors(adata, root_milestone = 'Root', milestones = ['NE', 'Late2'])

In [ ]:
scf.pl.slide_cors(adata, root_milestone = 'Root', milestones = ['NE', 'Late2'],
                  basis = 'mesencoder', win_keep = [22, 23], focus = 0)

In [ ]:
adata.uns['Root->NE<>Late2']['fork'].index.tolist()